In [ ]:
import pandas as pd
import numpy as np
import re, os
import time
from pathlib import Path
from itertools import product
from io import StringIO

from Bio import Entrez, SeqIO
from Bio.Seq import Seq

from dotenv import load_dotenv

In [ ]:
# Path to your secrets environment file
ENV_PATH = r"C:\Users\oozgen\API_KEYS\secrets.env"

# Load the environment variables from the specific path
load_dotenv(dotenv_path=ENV_PATH)

In [ ]:
# ============================================================
# 0. Load latest processed file from previous step
# ============================================================

INPUT_PATH = Path(
    r"C:\Users\oozgen\Desktop\phd\dataset\training\03_chemistry strings into position-level annotations\SOD1_process_v1.csv"
)

df = pd.read_csv(INPUT_PATH)
df.columns = df.columns.str.strip()

print("Loaded file:", INPUT_PATH)
print("Shape:", df.shape)

df.head()

In [ ]:
# ============================================================
# Output folders
# ============================================================

TRAINING_DIR = Path(r"C:\Users\oozgen\Desktop\phd\dataset\training")

STEP5_DIR = TRAINING_DIR / "04_inhibition_groups_and_leakage_safe_splits"
STEP6_DIR = TRAINING_DIR / "05_transcript_coordinate_mapping"
STEP7_DIR = TRAINING_DIR / "06_biophysical_features"
STEP8_DIR = TRAINING_DIR / "07_model_ready_dataset"

for d in [STEP5_DIR, STEP6_DIR, STEP7_DIR, STEP8_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Output folders created.")

In [ ]:
# Divide inhibition into four equal-sized groups and prevent leakage
# inhibition_group_4_equal_count
# split_2way
# split_3way

In [ ]:
# ============================================================
# 5A. Create four equal-count inhibition groups
# ============================================================

required_cols = ["ISIS", "Sequence", "Inhibition(%)"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for step 5: {missing_cols}")

df["Sequence"] = df["Sequence"].astype(str).str.upper().str.strip()
df["inhibition_pct"] = pd.to_numeric(df["Inhibition(%)"], errors="coerce")

if df["inhibition_pct"].isna().any():
    bad_rows = df.loc[df["inhibition_pct"].isna(), ["ISIS", "Sequence", "Inhibition(%)"]]
    display(bad_rows)
    raise ValueError("Some Inhibition(%) values could not be converted to numeric.")

# Rank-based qcut gives equal-sized groups even when inhibition values are duplicated.
df = df.sort_values(["inhibition_pct", "ISIS"]).reset_index(drop=True)
df["inhibition_rank_first"] = df["inhibition_pct"].rank(method="first")

df["inhibition_group_4_equal_count"] = pd.qcut(
    df["inhibition_rank_first"],
    q=4,
    labels=[
        "G1_lowest_inhibition",
        "G2_low_mid_inhibition",
        "G3_high_mid_inhibition",
        "G4_highest_inhibition",
    ],
)

group_summary = (
    df
    .groupby("inhibition_group_4_equal_count", observed=False)
    .agg(
        n_ASOs=("ISIS", "count"),
        min_inhibition=("inhibition_pct", "min"),
        max_inhibition=("inhibition_pct", "max"),
        mean_inhibition=("inhibition_pct", "mean"),
        median_inhibition=("inhibition_pct", "median"),
    )
)

group_summary

In [ ]:
# ============================================================
# 5B. Leakage-safe 2-way split: train / test
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold

X_dummy = np.zeros(len(df))
y_strat = df["inhibition_group_4_equal_count"].astype(str)
groups = df["Sequence"]

sgkf = StratifiedGroupKFold(
    n_splits=5,        # approximately 80% train, 20% test
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(sgkf.split(X_dummy, y_strat, groups))

df["split_2way"] = "train"
df.loc[test_idx, "split_2way"] = "test"

print("2-way split counts:")
print(df["split_2way"].value_counts())

print("\nInhibition-group distribution by 2-way split:")
display(pd.crosstab(
    df["split_2way"],
    df["inhibition_group_4_equal_count"],
    normalize="index"
))

# Leakage check
train_sequences = set(df.loc[df["split_2way"] == "train", "Sequence"])
test_sequences = set(df.loc[df["split_2way"] == "test", "Sequence"])
overlap = train_sequences.intersection(test_sequences)

print("\nSequence leakage check:")
print("Overlapping sequences between train and test:", len(overlap))

if len(overlap) > 0:
    raise ValueError("Sequence leakage detected between train and test.")

In [ ]:
# ============================================================
# 5D. Save step 5 output
# ============================================================

STEP5_OUT = STEP5_DIR / "SOD1_process_v2_inhibition_groups_and_splits.csv"
GROUP_SUMMARY_OUT = STEP5_DIR / "SOD1_inhibition_group_summary_v2.csv"

df.to_csv(STEP5_OUT, index=False)
group_summary.to_csv(GROUP_SUMMARY_OUT)

print("Saved:")
print(STEP5_OUT)
print(GROUP_SUMMARY_OUT)

In [ ]:
# Map ASOs back to SOD1 transcript coordinates

In [ ]:
# ============================================================
# 6A. Load latest file after inhibition grouping / split
# ============================================================

INPUT_PATH = Path(
    r"C:\Users\oozgen\Desktop\phd\dataset\training\04_inhibition_groups_and_leakage_safe_splits\SOD1_process_v2_inhibition_groups_and_splits.csv"
)

df = pd.read_csv(INPUT_PATH)
df.columns = df.columns.str.strip()

print("Loaded:", INPUT_PATH)
print("Shape:", df.shape)

df.head()

In [ ]:
# ============================================================
# Output folder for step 6
# ============================================================

TRAINING_DIR = Path(r"C:\Users\oozgen\Desktop\phd\dataset\training")

STEP6_DIR = TRAINING_DIR / "05_transcript_coordinate_mapping"
STEP6_DIR.mkdir(parents=True, exist_ok=True)

STEP6_OUT = STEP6_DIR / "SOD1_process_v3_transcript_mapping.csv"
MAPPING_ALL_HITS_OUT = STEP6_DIR / "SOD1_transcript_mapping_all_hits_v3.csv"
FETCHED_TRANSCRIPT_FASTA_OUT = STEP6_DIR / "SOD1_refseq_transcript_fetched_from_NCBI.fasta"

print("Step 6 output folder:")
print(STEP6_DIR)

In [ ]:
# ============================================================
# 6B. Configure NCBI Entrez
# ============================================================

Entrez.email = "202350100001@ogr.akdeniz.edu.tr"   # replace with your email
Entrez.tool = "SOD1_ASO_mapping_notebook"

# Option 1: paste your key only for this active notebook session
NCBI_API_KEY = os.getenv("NCBI_API_KEY")

Entrez.api_key = NCBI_API_KEY

if Entrez.email == "your.email@example.com":
    raise ValueError("Please replace Entrez.email with your real email address.")

if Entrez.api_key in [None, "", "PASTE_YOUR_NCBI_API_KEY_HERE"]:
    raise ValueError("Please provide your NCBI API key.")

print("Entrez configured with email, tool name, and API key.")

In [ ]:
# ============================================================
# 6C. Fetch SOD1 RefSeq transcript from NCBI
# ============================================================

REFSEQ_ACCESSION = "NM_000454"

# Search nucleotide database for the RefSeq accession
search_handle = Entrez.esearch(
    db="nucleotide",
    term=f"{REFSEQ_ACCESSION}[Accession]",
    retmode="xml"
)

search_record = Entrez.read(search_handle)
search_handle.close()

id_list = search_record.get("IdList", [])

print("NCBI search result IDs:", id_list)

if len(id_list) == 0:
    raise ValueError(f"No NCBI nucleotide record found for accession {REFSEQ_ACCESSION}.")

# Usually only one UID is expected for a specific RefSeq accession
ncbi_uid = id_list[0]

# Small delay is polite, although this workflow makes only a few requests
time.sleep(0.2)

# Fetch FASTA by UID
fetch_handle = Entrez.efetch(
    db="nucleotide",
    id=ncbi_uid,
    rettype="fasta",
    retmode="text"
)

fasta_text = fetch_handle.read()
fetch_handle.close()

record = SeqIO.read(StringIO(fasta_text), "fasta")

transcript_id = record.id
transcript_description = record.description
transcript_seq = str(record.seq).upper().replace("U", "T")

print("Fetched transcript:")
print("UID:", ncbi_uid)
print("Transcript ID:", transcript_id)
print("Description:", transcript_description)
print("Length:", len(transcript_seq))

# Save fetched FASTA for reproducibility
with open(FETCHED_TRANSCRIPT_FASTA_OUT, "w") as f:
    f.write(fasta_text)

print("Saved fetched FASTA:")
print(FETCHED_TRANSCRIPT_FASTA_OUT)

In [ ]:
# ============================================================
# 6D. Map ASOs to the SOD1 transcript
# ============================================================

required_cols = ["ISIS", "Sequence"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["Sequence"] = df["Sequence"].astype(str).str.upper().str.strip()

def find_all_exact_matches(query, reference):
    """
    Return all 0-based start positions where query occurs in reference.
    """
    hits = []
    start = reference.find(query)

    while start != -1:
        hits.append(start)
        start = reference.find(query, start + 1)

    return hits


def map_aso_to_transcript(aso_sequence, transcript_sequence):
    """
    Maps an ASO sequence to a transcript sequence.

    Expected biological orientation:
        reverse_complement(ASO) matches the mRNA transcript.

    Also checks direct orientation as a safeguard.
    """
    aso = str(aso_sequence).upper().replace("U", "T")
    aso_rc = str(Seq(aso).reverse_complement()).upper()

    records = []

    # Expected antisense orientation
    rc_hits = find_all_exact_matches(aso_rc, transcript_sequence)

    for start in rc_hits:
        records.append({
            "target_start_0based": start,
            "target_end_0based_exclusive": start + len(aso_rc),
            "target_start_1based": start + 1,
            "target_end_1based_inclusive": start + len(aso_rc),
            "matched_transcript_sequence": aso_rc,
            "mapping_orientation": "ASO_reverse_complement_matches_transcript",
        })

    # Direct orientation safeguard
    direct_hits = find_all_exact_matches(aso, transcript_sequence)

    for start in direct_hits:
        records.append({
            "target_start_0based": start,
            "target_end_0based_exclusive": start + len(aso),
            "target_start_1based": start + 1,
            "target_end_1based_inclusive": start + len(aso),
            "matched_transcript_sequence": aso,
            "mapping_orientation": "ASO_direct_sequence_matches_transcript",
        })

    return records


mapping_records = []

for _, row in df.iterrows():
    hits = map_aso_to_transcript(row["Sequence"], transcript_seq)

    row_id_value = row["row_id"] if "row_id" in df.columns else pd.NA

    if len(hits) == 0:
        mapping_records.append({
            "row_id": row_id_value,
            "ISIS": row["ISIS"],
            "Sequence": row["Sequence"],
            "transcript_source": "NCBI_RefSeq",
            "ncbi_uid": ncbi_uid,
            "transcript_id": transcript_id,
            "transcript_description": transcript_description,
            "transcript_length": len(transcript_seq),
            "n_transcript_hits": 0,
            "target_start_0based": pd.NA,
            "target_end_0based_exclusive": pd.NA,
            "target_start_1based": pd.NA,
            "target_end_1based_inclusive": pd.NA,
            "matched_transcript_sequence": pd.NA,
            "mapping_orientation": "no_exact_match",
        })
    else:
        for hit in hits:
            mapping_records.append({
                "row_id": row_id_value,
                "ISIS": row["ISIS"],
                "Sequence": row["Sequence"],
                "transcript_source": "NCBI_RefSeq",
                "ncbi_uid": ncbi_uid,
                "transcript_id": transcript_id,
                "transcript_description": transcript_description,
                "transcript_length": len(transcript_seq),
                "n_transcript_hits": len(hits),
                **hit,
            })

mapping_df = pd.DataFrame(mapping_records)

print("Mapping status:")
print(mapping_df["mapping_orientation"].value_counts(dropna=False))

mapping_df.head()

In [ ]:
# ============================================================
# 6E. Merge first transcript-mapping hit into ASO-level dataset
# ============================================================

mapping_first_hit = (
    mapping_df
    .sort_values(
        by=["ISIS", "n_transcript_hits", "target_start_0based"],
        ascending=[True, False, True],
        na_position="last"
    )
    .drop_duplicates("ISIS")
)

mapping_cols_to_merge = [
    "ISIS",
    "transcript_source",
    "ncbi_uid",
    "transcript_id",
    "transcript_description",
    "transcript_length",
    "n_transcript_hits",
    "target_start_0based",
    "target_end_0based_exclusive",
    "target_start_1based",
    "target_end_1based_inclusive",
    "matched_transcript_sequence",
    "mapping_orientation",
]

# Drop old mapping columns if rerunning this cell
old_mapping_cols = [c for c in mapping_cols_to_merge if c != "ISIS"]
df = df.drop(columns=[c for c in old_mapping_cols if c in df.columns], errors="ignore")

df = df.merge(
    mapping_first_hit[mapping_cols_to_merge],
    on="ISIS",
    how="left",
    validate="one_to_one"
)

df["transcript_mapping_attempted"] = True

print("Merged dataset shape:", df.shape)
print()
print(df["mapping_orientation"].value_counts(dropna=False))

df[
    [
        "ISIS",
        "Sequence",
        "transcript_id",
        "n_transcript_hits",
        "target_start_1based",
        "target_end_1based_inclusive",
        "mapping_orientation",
    ]
].head()

In [ ]:
# ============================================================
# 6F. Save Step 6 outputs
# ============================================================

df.to_csv(STEP6_OUT, index=False)
mapping_df.to_csv(MAPPING_ALL_HITS_OUT, index=False)

print("Saved:")
print(STEP6_OUT)
print(MAPPING_ALL_HITS_OUT)
print(FETCHED_TRANSCRIPT_FASTA_OUT)